In [0]:
dbutils.library.restartPython()

In [0]:
"""Benchmark fetch_hourly_forcast over batch size and concurrency combinations."""

import asyncio
import itertools
import logging
import math
import random
import statistics
import time
from dataclasses import dataclass

from open_meteo import Location, fetch_hourly_forecast

N_LOCATIONS = 100
BATCH_SIZES = (20, 45, 50, 75, 100)
CONCURRENCIES = (1, 2, 4, 8, 16)
REPEATS = 3
COOLDOWN_S = 15  # keeps the per-minute request rate under the API limit
SEED = 42
SESSION_SALT = random.SystemRandom().randrange(2**32)  # fresh coordinates every session

logging.basicConfig(level=logging.WARNING)


@dataclass(frozen=True)
class Combo:
    """One batch size and concurrency pair under test."""

    batch_size: int
    concurrency: int


@dataclass(frozen=True)
class Run:
    """Outcome of one benchmark run."""

    combo: Combo
    seconds: float
    received: int
    expected: int

    @property
    def complete(self) -> bool:
        return self.received == self.expected


def make_locations(run_index: int, n: int = N_LOCATIONS) -> list[Location]:
    """Random coordinates, unique per session and run, so cached responses can't skew timings."""
    rng = random.Random(SESSION_SALT + run_index)
    return [
        Location(round(rng.uniform(-60, 70), 2), round(rng.uniform(-180, 180), 2))
        for _ in range(n)
    ]


def valid_combos() -> list[Combo]:
    """Skip combinations where concurrency exceeds the number of batches."""
    return [
        Combo(batch_size, concurrency)
        for batch_size, concurrency in itertools.product(BATCH_SIZES, CONCURRENCIES)
        if concurrency <= math.ceil(N_LOCATIONS / batch_size)
    ]


async def run_once(combo: Combo, run_index: int, n: int = N_LOCATIONS) -> Run:
    """Time one full fetch for the given combination."""
    locations = make_locations(run_index, n)
    start = time.perf_counter()
    try:
        results = await fetch_hourly_forecast(
            locations,
            batch_size=combo.batch_size,
            max_concurrency=combo.concurrency,
        )
    except Exception:
        logging.exception("run failed for %s", combo)
        results = []
    return Run(combo, time.perf_counter() - start, len(results), len(locations))


async def smoke_test() -> None:
    """Fetch 10 locations in one request and stop if it does not fully succeed."""
    run = await run_once(Combo(batch_size=10, concurrency=1), run_index=-1, n=10)
    if not run.complete:
        raise SystemExit(f"smoke test failed: got {run.received}/{run.expected}")
    print(f"smoke test ok ({run.seconds:.2f}s for 10 locations)")


def summarise(runs: list[Run]) -> None:
    """Print one row per combination, fastest first. Only complete runs are timed."""
    rows = []
    for combo in dict.fromkeys(r.combo for r in runs):
        mine = [r for r in runs if r.combo == combo]
        good = [r.seconds for r in mine if r.complete]
        if len(good) == 0:
            rows.append((math.inf, combo, 0, len(mine), math.nan, math.nan))
            continue
        median = statistics.median(good)
        rows.append((median, combo, len(good), len(mine), min(good), N_LOCATIONS / median))
    print(f"\n{'batch':>6} {'conc':>5} {'ok/runs':>8} {'median s':>9} {'min s':>7} {'loc/s':>7}")
    for median, combo, ok, total, fastest, rate in sorted(rows, key=lambda r: r[0]):
        print(
            f"{combo.batch_size:>6} {combo.concurrency:>5} {ok:>4}/{total:<3} "
            f"{median:>9.2f} {fastest:>7.2f} {rate:>7.1f}"
        )


async def main() -> None:
    assert all(N_LOCATIONS % b != 1 for b in BATCH_SIZES), "avoid a final batch of one location"
    print(f"session salt: {SESSION_SALT}")
    await smoke_test()
    plan = [(c, r) for c in valid_combos() for r in range(REPEATS)]
    random.Random(SEED).shuffle(plan)  # shuffled so time-of-day drift doesn't favour a combo
    runs: list[Run] = []
    for i, (combo, _) in enumerate(plan):
        run = await run_once(combo, i)
        print(f"[{i + 1}/{len(plan)}] {combo} {run.seconds:.2f}s {run.received}/{run.expected}")
        runs.append(run)
        await asyncio.sleep(COOLDOWN_S)
    summarise(runs)

In [0]:
await main()

In [0]:
%sql
drop table open_meteo.bronze_hourly_forecast;
drop table open_meteo.dim_city;
drop table open_meteo.silver_hourly_forecast;

In [0]:
%sql
select * from open_meteo.bronze_hourly_forecast;


In [0]:
%sql
select * from open_meteo.dim_city;

In [0]:
%sql
select * from open_meteo.silver_hourly_forecast;

In [0]:
%sql
select * from open_meteo.silver_hourly_forecast_rejects;